# Deploy Lakehouse

In [1]:
from pathlib import Path
from trino_stack.render import render_collective
from trino_stack.lakehouse import Lakehouse
import trino_stack

"""
Deploy lakehouse with the tpcds schema and register to HMS
"""

TRINO_STACK_DIR = Path(trino_stack.__file__).resolve().parent

res = render_collective(
    templates_dir=str(TRINO_STACK_DIR / "manifests"),
    values_path=str(TRINO_STACK_DIR / "lakehouses" / "lakehouse-tpcds-small.yaml"),
)

lh = Lakehouse(res, verbose=False)

lh.deploy()
lh.register_schema("tpcds", "/mnt/iceberg/warehouse", use_dot_db=True)  # <schema name in /warehouse>  <location of /warehouse in control container>


Deploying Trino
Checking Trino health ...
STATUS:

Lakehouse: lakehouse-g
Namespace: pgr24james
Selector:  app.kubernetes.io/instance=lakehouse-g
URL:       http://trino-route-lakehouse-g-pgr24james.apps.os.dcs.gla.ac.uk

Pods:

  Coordinator:
    - trino-coord-pod-lakehouse-g              Running    Ready     idagpu-22          10.130.13.132

  Metastore:
    - hive-metastore-postgres-lakehouse-g      Running    Ready     idagpu-head        10.129.2.87

  Worker:
    - trino-worker-lakehouse-g-0-0             Running    Ready     idagpu-22          10.130.13.131

Schemas:
  <none>
Waiting for Trino engine to ready.
Trino is ready.
Trino host: trino-service-lakehouse-g.pgr24james.svc.cluster.local
Schema:     tpcds
Warehouse:  /mnt/iceberg/warehouse
Tables:     24
  FAILED call_center: TrinoUserError(type=USER_ERROR, name=ALREADY_EXISTS, message="Table already exists: 'tpcds.call_center'", query_id=20260725_124914_02063_vhz6g)
  FAILED catalog_page: TrinoUserError(type=USER_ERROR, name

# Generate Queries

In [1]:
import logging; logging.getLogger("httpx").setLevel(logging.WARNING)

In [9]:

"""
Generate a query workload using the OpenAI API 
"""
from trino_stack.lakehouse import Lakehouse
lh = Lakehouse.from_release(instance_name="lakehouse-g", namespace="pgr24james",sync_schemas=True, verbose=True)


report = lh.generate_workload(
    schema="tpcds",
    workload_name="gpt_generated_1000_tpcds_tokens_no_feedback",
    num_queries=1000,
    min_tables=2,
    max_tables=16,
    warmup=True,
    validate=True,
    batch_size=100,
    generation_workers=64,
    validation_workers=8,
    reasoning="high",
    request_timeout_s=1200,
    plan_feedback=False,
)


Checking Trino health ...
STATUS:

Lakehouse: lakehouse-g
Namespace: pgr24james
Selector:  app.kubernetes.io/instance=lakehouse-g
URL:       http://trino-route-lakehouse-g-pgr24james.apps.os.dcs.gla.ac.uk

Pods:

  Coordinator:
    - trino-coord-pod-lakehouse-g              Running    Ready     idagpu-22          10.130.13.132

  Metastore:
    - hive-metastore-postgres-lakehouse-g      Running    Ready     idagpu-head        10.129.2.87

  Worker:
    - trino-worker-lakehouse-g-0-0             Running    Ready     idagpu-22          10.130.13.131

Schemas:

  Warehouse location: /mnt/iceberg/warehouse

  tpcds: 24 tables
    - call_center
    - catalog_page
    - catalog_returns
    - catalog_sales
    - customer
 	 ... plus 19 more
Model warm-up response: ready
Generating 1000 queries (adaptive continuous pool, ceiling workers=64)


gen gpt_generated_1000_tpcds_tokens_no_feedback:   0%|          | 0/1000 [00:00<?, ?q/s]

[novelty control] rejected 438 invalid, 0 sql-duplicates, 146 plan-duplicates, 0 plan-capped, 4 near-duplicates, 35 skeleton-capped (1669/4000 attempts for 1000 queries)
Finished query generation
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...


# Config

In [3]:
import json
from trino_stack.config import MODEL_NAME, BASE_MODEL_URL, API_KEY_ENV
from workload_generation.baselines.common import context_from_lakehouse

"""
Hardcoded config used by every generation cell below.
Edit these values to change schema / instance / model / volume.
"""

INSTANCE_NAME = "lakehouse-g"
NAMESPACE = "pgr24james"

SCHEMA = "tpcds"
CATALOG = "iceberg"

NUM_QUERIES = 1000

TEMPERATURE = 0.6
REASONING = "high"

RANDOM_SEED = 42
WARMUP = True

ctx = context_from_lakehouse(
    lh,
    schema=SCHEMA,
    catalog=CATALOG,
    model_name=MODEL_NAME,
    base_url=BASE_MODEL_URL,
    api_key_env=API_KEY_ENV,
    temperature=TEMPERATURE,
    reasoning=REASONING,
)


# Generate Baseline Workloads

## sqlstorm

In [4]:
from workload_generation.baselines import sqlstorm

"""
Generate a workload using the sqlstorm baseline
"""

WORKLOAD_NAME_SQLSTORM = "sqlstorm_1000_tokens_{}".format(SCHEMA)

report_sqlstorm = sqlstorm.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_SQLSTORM,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_sqlstorm["baseline"],
    "workload_dir": report_sqlstorm["workload_dir"],
    "num_queries": report_sqlstorm["num_queries"],
    "duration_s": round(report_sqlstorm["duration_s"], 1),
}, indent=2, default=str))


Model warm-up response: ready
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 2/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerE

2026-08-07 03:55:05,378 [INFO] Applying array index offset (-1)
2026-08-07 03:55:05,380 [INFO] Applying array index offset (-1)
2026-08-07 03:55:10,057 [INFO] Applying array index offset (-1)
2026-08-07 03:55:13,905 [INFO] Applying array index offset (-1)
2026-08-07 03:55:14,578 [INFO] Applying array index offset (-1)
2026-08-07 03:55:14,580 [INFO] Applying array index offset (-1)
2026-08-07 03:55:14,581 [INFO] Applying array index offset (-1)
2026-08-07 03:55:14,582 [INFO] Applying array index offset (-1)
2026-08-07 03:55:14,583 [INFO] Applying array index offset (-1)
2026-08-07 03:55:20,380 [INFO] Applying array index offset (-1)
2026-08-07 03:55:23,171 [INFO] Applying array index offset (-1)
2026-08-07 03:55:23,173 [INFO] Applying array index offset (-1)
2026-08-07 03:55:24,912 [INFO] Applying array index offset (-1)
2026-08-07 03:55:24,915 [INFO] Applying array index offset (-1)
2026-08-07 03:55:24,916 [INFO] Applying array index offset (-1)
2026-08-07 03:55:24,918 [INFO] Applying 

{
  "baseline": "sqlstorm",
  "workload_dir": "/mnt/primary/Main/Workloads/sqlstorm_1000_tokens_tpcds",
  "num_queries": 1000,
  "duration_s": 53995.1
}


## sqlbarber

In [5]:
from workload_generation.baselines import sqlbarber

"""
Generate a workload using the sqlbarber baseline
"""

WORKLOAD_NAME_SQLBARBER = "sqlbarber_1000_tokens_{}".format(SCHEMA)

report_sqlbarber = sqlbarber.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_SQLBARBER,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_sqlbarber["baseline"],
    "workload_dir": report_sqlbarber["workload_dir"],
    "num_queries": report_sqlbarber["num_queries"],
    "duration_s": round(report_sqlbarber["duration_s"], 1),
}, indent=2, default=str))


Model warm-up response: ready
{
  "baseline": "sqlbarber",
  "workload_dir": "/mnt/primary/Main/Workloads/sqlbarber_1000_tokens_tpcds",
  "num_queries": 1000,
  "duration_s": 11629.9
}


## e2etune

In [6]:
from workload_generation.baselines import e2etune

"""
Generate a workload using the e2etune baseline
"""

WORKLOAD_NAME_E2ETUNE = "e2etune_1000_tokens_{}".format(SCHEMA)

report_e2etune = e2etune.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_E2ETUNE,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_e2etune["baseline"],
    "workload_dir": report_e2etune["workload_dir"],
    "num_queries": report_e2etune["num_queries"],
    "duration_s": round(report_e2etune["duration_s"], 1),
}, indent=2, default=str))


Model warm-up response: ready
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
{
  "baseline": "e2etune",
  "workload_dir": "/mnt/primary/Main/Workloads/e2etune_1000_tokens_tpcds",
  "num_queries": 1000,
  "duration_s": 14372.5
}


## bootstrapping_lcm

In [7]:
from workload_generation.baselines import bootstrapping_lcm

"""
Generate a workload using the bootstrapping_lcm baseline
"""

WORKLOAD_NAME_BOOTSTRAPPING_LCM = "bootstrapping_lcm_1000_tokens_{}".format(SCHEMA)

report_bootstrapping_lcm = bootstrapping_lcm.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_BOOTSTRAPPING_LCM,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_bootstrapping_lcm["baseline"],
    "workload_dir": report_bootstrapping_lcm["workload_dir"],
    "num_queries": report_bootstrapping_lcm["num_queries"],
    "duration_s": round(report_bootstrapping_lcm["duration_s"], 1),
}, indent=2, default=str))


Model warm-up response: ready
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerError: Error code: 504 - {'error': 'Request timeout'}
Sleeping 2.5s before retry...
[API retry 1/2000] InternalServerE

## sql_factory

In [8]:
from workload_generation.baselines import sql_factory

"""
Generate a workload using the sql_factory baseline.

"""

WORKLOAD_NAME_SQL_FACTORY = "sql_factory_1000_tokens_{}".format(SCHEMA)

report_sql_factory = sql_factory.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_SQL_FACTORY,
    num_queries=NUM_QUERIES,
    queries_per_call=4,     
    max_stall_rounds=10,     
    saturation_stop=False,   
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_sql_factory["baseline"],
    "workload_dir": report_sql_factory["workload_dir"],
    "num_queries": report_sql_factory["num_queries"],
    "duration_s": round(report_sql_factory["duration_s"], 1),
    "stop_reason": report_sql_factory["pipeline"]["termination"]["stop_reason"],
}, indent=2, default=str))


Model warm-up response: ready
{
  "baseline": "sql_factory",
  "workload_dir": "/mnt/primary/Main/Workloads/sql_factory_1000_tokens_tpcds",
  "num_queries": 1000,
  "duration_s": 19919.7,
  "stop_reason": "target_reached"
}


# Tear Down Lakehouse

In [2]:
"""
Tear down the lakehouse deployed in the first cell
"""

lh.tear_down()


Successfully tore down lakehouse lakehouse-g
